# 🏫 HW1: Exploratory Data Analysis

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw01.ipynb")

You must submit this assignment to Gradescope by the on-time deadline, **Friday, September 11th, 11:59 PM**. Please read the syllabus for the Slip Day policy. No late submissions beyond the details in the Slip Day policy will be accepted. While course staff is happy to help you if you encounter difficulties with submission, we may not be able to respond to late-night requests for assistance (TAs need to sleep, after all!). **We strongly encourage you to plan to submit your work to Gradescope several hours before the stated deadline.** This way, you will have ample time to contact staff for submission support. 

After completing this part ("Homework 1 Coding"), please submit the generated zip file to the Homework 1 Coding assignment on Gradescope. Gradescope will automatically submit a PDF of your written responses to the HW 1 Coding Written assignment; there is no need to submit it manually.

## 💪 Collaboration Policy

Data science is a collaborative activity. While you may talk with others about
the homework, we ask that you **write your solutions individually**. If you 
discuss the assignments with others, please **include their names** below.

**Collaborator(s)**: *List all collaborator(s) here*.

## 💯 Score Breakdown
Question | Manual? | Points
--- | --- | --
1a | no | 1
1b | no | 1
2a | no | 1
2b | no | 2
2ci | no | 1
2cii | no | 1
2d | yes | 1
2e | no | 2
2f | no | 1
3a | no | 1
3b | no | 2
3c | no | 0
4a | no | 2
4b | no | 2
5 | yes | 5
6 | yes | 3
Total |   | 26

**Note**: "Manual" questions are written response questions that will be graded manually by the grading team instead of being graded by the autograder.

## ✊ Before You Start

### Autograder and Answer Cells

For each question in the assignment, please write down your answer in the answer cell(s) right below the question. 

We understand that it is helpful to have extra cells breaking down the process toward reaching your final answer. If you happen to create new cells *below* your answer to run code, **NEVER** add cells between a question cell and the answer cell below it. It may cause errors when we run the autograder, and it will cause a failure to generate the PDF file.



### Initialize your environment

The below cell should run without error if you're using the course DataHub.

In [ ]:
# DO NOT EDIT THIS CELL
import numpy as np
import polars as pl

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
plt.style.use('fivethirtyeight')

from IPython.display import display, Image 
def display_figure_for_grader(fig):
    plotly.io.write_image(fig, 'temp.png')
    display(Image('temp.png'))

## 🐻‍❄️ Part 1: Cleaning and Exploring Data with Polars

*For most of Part 1, we recommend minimal to no LLM usage, so that you get practice with the basics of `polars`. A few subparts will be marked as LLM-OK, and you are encouraged to use LLMs and/or documentation searches to help you with those questions, as long as you follow the [course LLM policy](https://ds100.org/fa26/syllabus/#collaboration-policy-and-academic-honesty).*

### Context

In the first part of this assignment, we will investigate restaurant food safety scores for restaurants in San Francisco. The scores and violation information have been [made available by the San Francisco Department of Public Health](https://data.sfgov.org/Health-and-Social-Services/Restaurant-Scores-LIVES-Standard/pyih-qa8i).

After this subpart, you should be comfortable with:
* Reading CSV files, 
* Reading `polars` documentation and using `polars`,
* Working with data at different levels of granularity, and
* Identifying the type of data collected, missing values, anomalies, etc., and doing some basic analysis.

### Reading in and Verifying Data

Let's attempt to load `bus.csv`, `ins2vio.csv`, `ins.csv`, and `vio.csv` into `Polars` `DataFrame`s with the following names: `bus`, `ins2vio`, `ins`, and `vio`, respectively. You'll explore what each one contains once you get to question 1.

*Note:* Because of character encoding issues, one of the files (`bus`) will require an additional argument `encoding='latin-1'` when calling `pl.read_csv`. At some point in your future, you should read all about [character encodings](https://diveintopython3.net/strings.html). We won't discuss these in detail in Data 100.

In [ ]:
# Path to the directory containing data
from pathlib import Path
dsDir = Path('data')

bus = pl.read_csv(dsDir/'bus.csv', encoding='latin-1')
ins2vio = pl.read_csv(dsDir/'ins2vio.csv')
ins = pl.read_csv(dsDir/'ins.csv')
vio = pl.read_csv(dsDir/'vio.csv')

# This code is essential for the autograder to function properly. Do not edit
# Notice that we can use pl.col with multiple columns: this is convenient
# when we want to apply an operation like round(2) to multiple columns at once
bus = bus.with_columns(pl.col("latitude", "longitude").round(2))
ins_test = ins

Now that you've read the files let's try some `pl.DataFrame` methods ([docs](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html)).
Use the `DataFrame.head` method to show the top few lines of the `bus`, `ins`, and `vio` `DataFrame`s. For example, running the cell below will display the first few lines of the `bus` `DataFrame`.

In [ ]:
bus.head()

To show multiple return outputs in one single cell, you can use `display()`.

In [ ]:
display(bus.head())
display(ins.head())

The `DataFrame.describe` method can also be handy for computing summaries of numeric columns of our `DataFrame`s. Try it out with each of our 4 `DataFrame`s. Below, we have used the method to give a summary of the `bus` `DataFrame`.

In [ ]:
bus.describe()

Now, we perform some sanity checks for you to verify that the data was loaded with the correct structure.

First, we check the basic structure of the `DataFrame`s you created:

In [ ]:
assert bus.columns == ['business id column', 'name', 'address', 'city', 'state', 'postal_code',
                       'latitude', 'longitude', 'phone_number'] # Check that the `bus` DataFrame contains the right columns in the correct order
assert 6250 <= len(bus) <= 6260 # Check that the `bus` DataFrame has the correct number of rows

assert ins.columns == ['iid', 'date', 'score', 'type'] # Similar to above, check that the `ins` DataFrame contains the right columns in order
assert 26660 <= len(ins) <= 26670 # Similar to above, check that the `ins` DataFrame has the correct number of rows

assert vio.columns == ['description', 'risk_category', 'vid']
assert 60 <= len(vio) <= 65

assert ins2vio.columns == ['iid', 'vid']
assert 40210 <= len(ins2vio) <= 40220

*Fun Fact*: The series of `assert` statements above is similar to how course staff implement test cases for assignments! Next time you fail a public test case on an assignment, take a look at the error message and you will probably find the `assert` statement which is causing you to fail the test case.



Next we'll check that the statistics match what we expect. The following are hard-coded statistical summaries of the correct data.

In [ ]:
bus_summary = pl.DataFrame({
 'statistic': ['min', '50%', 'max'],
 'business id column': [19.0, 75685.0, 102705.0],
 'latitude': [-9999.0, -9999.0, 37.82],
 'longitude': [-9999.0, -9999.0, 0.0]})

ins_summary = pl.DataFrame({
 'statistic': ['min', '50%', 'max'],
 'score': [-1.0, 76.0, 100.0]})

vio_summary = pl.DataFrame({
 'statistic': ['min', '50%', 'max'],
 'vid': [103102.0, 103135.0, 103177.0]})

from IPython.display import display

print('What we expect from your Businesses DataFrame:')
display(bus_summary)
print('What we expect from your Inspections DataFrame:')
display(ins_summary)
print('What we expect from your Violations DataFrame:')
display(vio_summary)

The code below defines a testing function that we'll use to verify that your data has the same statistics as what we expect. Run these cells to define the function. The `df_allclose` function has this name because we are verifying that all of the statistics for your `DataFrame` are close to the expected values. Why not `df_allequal`? It's a bad idea in almost all cases to compare two floating point values like 37.780435, as rounding errors can cause spurious failures. Run the following cells to load some basic utilities (you do not need to change these at all):

In [ ]:
"""Run this cell to load this utility comparison function that we will use in various
tests below (both tests you can see and those we run internally for grading).

Do not modify the function in any way.
"""


def summary_stats(df, columns):
    """Return the min, median, and max of the given columns as one row per statistic."""
    return pl.DataFrame({
        'statistic': ['min', '50%', 'max'],
        **{c: [df[c].min(), df[c].median(), df[c].max()] for c in columns}
    })

def df_allclose(actual, desired, columns=None, rtol=5e-2):
    """Compare selected columns of two DataFrames on a few summary statistics.

    Compute the min, median, and max of the two DataFrames on the given columns, and compare
    that they match numerically to the given relative tolerance.
    """
    # For the desired values, we can provide a full DF with the same structure as
    # the actual data or pre-computed summary statistics.
    # We assume a pre-computed summary was provided if `columns` is None. In that case,
    # `desired` *must* have the same structure as the actual's summary
    if columns is None:
        des = desired
        columns = [c for c in desired.columns if c != 'statistic']
    else:
        des = summary_stats(desired, columns)

    # Extract summary stats from actual DF
    act = summary_stats(actual, columns)

    return np.allclose(act.select(columns).to_numpy(), des.select(columns).to_numpy(), rtol)

We will now explore each file in turn, including determining its granularity and exploring many of the variables individually. Let's begin with the businesses file, which has been read into the `bus` `DataFrame`.

<br/>

---

<br/>

## 📚 1: Examining the Business Data File

### 🚌 Question 1a

From its name alone, we expect the `bus.csv` file to contain information about the restaurants (`bus`inesses). Let's investigate the granularity of this dataset.

In [ ]:
bus.head()

The `bus` `DataFrame` contains a column called `business id column`, which probably corresponds to a unique business id.  However, we will first rename that column to `bid` for simplicity.

**Note**: In practice, we might want to do this renaming when the table is loaded, but for grading purposes, we will do it here.

In [ ]:
bus = bus.rename({"business id column": "bid"})

Examining the entries in `bus`, is the `bid` unique for each record (i.e., each row of data)? Your code should compute the answer, i.e., don't just hard code `True` or `False`.

**Hint**: Use [`unique()` (documentation here)](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.unique.html) or [`value_counts()` (documentation here)](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.value_counts.html) to determine if the `bid` series has any duplicates.

**Hint**: for this question, just like many other questions, it can be helpful to create some new cells above the question to play around with the dataframe and the suggested methods. Try it now using the "+" button in the toolbar above! **Important: always be careful to never add cells between the question cell and the answer cell below it!**

In [ ]:
is_bid_unique = ...

is_bid_unique

In [ ]:
grader.check("q1a")

<br/>

---


### 🚌 Question 1b

Based on the above exploration, what does each record in the `bus` `DataFrame` represent?

**A**. A city block.

**B**. A chain of restaurants.

**C**. One location of a restaurant.

**D**. A postal code.


Answer in the following cell. Your answer should be a string, either `"A"`, `"B"`, `"C"`, or `"D"`.

In [ ]:
q1b = ...

In [ ]:
grader.check("q1b")

<br/>

---

<br/>

## 🧹 2: Cleaning the Business Data Postal Codes

The business data contains postal code information that we can use to aggregate the ratings over regions of the city. Let's examine and clean the postal code field. The postal code (sometimes also called a [ZIP code](https://en.wikipedia.org/wiki/ZIP_Code)) partitions the city into regions:

<img src="https://gisgeography.com/wp-content/uploads/2023/07/San-Francisco-Zip-Code-Map-1-2048x2048.jpg" alt="ZIP Code Map" style="width: 600px">

<br/>

---


### 🥡 Question 2a

How many restaurants are in each ZIP code? 

In the cell below, create a **DataFrame** with one row per postal code: a `postal_code` column, and a `count` column holding the number of records with that postal code. The `DataFrame` should be in descending order of count. Do you notice any odd/invalid ZIP codes?

In [ ]:
zip_counts = ...

# Using pl.Config(tbl_rows=-1) tells polars to display all the rows and not hide the ones in the middle.
# This can be useful when doing EDA, but be careful when working with very large dataframes!
with pl.Config(tbl_rows=-1):
    display(zip_counts)

In [ ]:
grader.check("q2a")

<br/>

--- 

### 🥡 Question 2b

In Question 2a, we noticed a large number of potentially invalid ZIP codes (e.g., "Ca"). These are likely due to data entry errors. To get a better understanding of the potential errors in the zip codes, let's break down the problem into two parts.

First, import a list of valid San Francisco ZIP codes by loading the file `data/sf_zipcodes.json`. This file is in a different file format called JSON: this is a convenient format that typically looks a lot like a Python dictionary. We have (at least) two options for how to read this file:

* The `json.load` function from the Python `json` library will load in data from a JSON file as a python dictionary.
* The `pl.read_json` file will read the data into a polars series or dataframe: this works similarly to `pl.read_csv`.

Since we won't need to do any tabular operations using this data, we'll just use `json.load`.

In [ ]:
import json
with open("data/sf_zipcodes.json") as f:
    zip_code_dict = json.load(f)
valid_zip_codes = zip_code_dict['zip_codes']
valid_zip_codes

Construct a `DataFrame` containing only the businesses that **DO NOT** have valid ZIP codes. You will probably want to use the `Series.is_in` function. For more information on this function see the [documentation](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.is_in.html). 

**Note:** You are **always** welcome and, in fact, encouraged to search and read the documentation on the internet to complete the assignments in the course, even if the documentation is not linked explicitly. Using LLMs to help you navigate documentation counts as an **appropriate use** under the course policy.

In [ ]:
...
invalid_zip_bus = ...
invalid_zip_bus.head(20)

In [ ]:
grader.check("q2b")

<br/>

--- 

### ❓ Question 2c

In the previous question, many of the businesses had a common invalid postal code that was likely used to encode a MISSING postal code. Do they all share a potentially "interesting address"? For that purpose, in the following cells, we will construct a series that counts the number of businesses at each `address` that have this single likely MISSING postal code value. 

Let's break this down into steps: 

#### ❓ Part I
Identify the single most common invalid postal code and assign it to `missing_postal_code`. Then create a `DataFrame`, `bus_missing`, to store only those businesses in `bus` that have `missing_postal_code` as their postal code.

**Hint**: All ZIP codes in the US are *positive* numbers

In [ ]:
missing_postal_code = ...
bus_missing = ...

In [ ]:
grader.check("q2ci")

#### ❓ Part II
Using `bus_missing`, find the number of businesses at each address (which would all share the same postal code). Specifically, `missing_zip_address_count` should store a `DataFrame` with an `address` column and a `count` column, one row per address, in descending order of count.

In [ ]:
missing_zip_address_count = ...
missing_zip_address_count.head()

In [ ]:
grader.check("q2cii")

<!-- BEGIN QUESTION -->

<br/>

--- 

### ❓ Question 2d

**(Written Response)** If we were to drop businesses with postal code values equal to `missing_postal_code`, what **specific types of businesses** would we be excluding? In other words, is there a commonality among businesses with missing postal codes? **Please respond in no more than 2 sentences.**

**Hint**: You may want to identify and Google the names of the businesses with missing postal codes. Feel free to reuse parts of your code from 2c to re-examine `bus_missing` but we will not be grading your code.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

--- 

### ❓ Question 2e

Examine the `invalid_zip_bus` `DataFrame` we computed in Question 2b and look at the businesses that DO NOT have the special MISSING ZIP code value. Some of the invalid postal codes are just the full 9-digit code rather than the first 5 digits. **Create a new column named `postal5` in the original `bus` `DataFrame`, which contains only the first 5 digits of the `postal_code` column.**

Then, for any of the `postal5` ZIP code entries that were not a valid San Francisco ZIP code (according to `valid_zip_codes`), the provided code will set the `postal5` value to `None`. 

**Hint:** You will find string expressions (the `.str` namespace) particularly useful. They allow you to use your usual Python string operations in tandem with a `DataFrame`. Refer to the [Polars string expression documentation](https://docs.pola.rs/api/python/stable/reference/expressions/string.html) for examples.

This question (2e) is a good candidate to try using an LLM to help you navigate the string operations mentioned above, but make sure you know exactly what's happening in the resulting code!
    
**Do not modify the provided code! Simply add your own code in place of the ellipses.** You might find it useful to understand how `pl.when().then().otherwise()` works, but you won't be required to know it for Data 100.


In [ ]:
#bus = bus.with_columns(pl.lit(None, dtype=pl.String).alias('postal5'))
...

# pl.when(<CONDITION>).then(<YES>).otherwise(<NO>) is an expression for doing if-then operations in polars:
# for each row, if the <CONDITION> is true, then that row gets whatever's in <YES>; otherwise it gets whatever's
# in <NO>.
bus = bus.with_columns(
    pl.when(pl.col('postal5').is_in(valid_zip_codes))
      .then(pl.col('postal5'))
      .otherwise(None)
      .alias('postal5')
)
# Checking the corrected postal5 column
bus.filter(~pl.col('postal_code').is_in(valid_zip_codes)).select(['bid', 'name', 'postal_code', 'postal5'])

In [ ]:
grader.check("q2e")

<br/>

---

<br/>

### ❓ Question 2f

Finally, use the `postal5` column to create a `DataFrame`, `bus_valid`, that only contains the rows of `bus` where a `postal5` zip code exists. You may find the `.is_not_null()` expression useful here.

In [ ]:
bus_valid = ...
bus_valid

In [ ]:
grader.check("q2f")

<br/>

---

<br/>

## 🔍 3: Investigate the Inspection Data

Let's now turn to the inspection `DataFrame`. Earlier, we found that `ins` has 4 columns named 
`iid`, `score`, `date`, and `type`.  In this section, we determine the granularity of `ins` and investigate the kinds of information provided for the inspections.

Let's start by looking again at the first 5 rows of `ins` to see what we're working with.

In [ ]:
ins.head(5)

<br/>

---

### 🔍 Question 3a

The column `iid` probably corresponds to an inspection ID. Write an expression (i.e., a line of code) that evaluates to `True` or `False` based on whether all the inspection IDs are unique. Your code should compute the answer, i.e., don't just hard code `True` or `False`.

**Hint:** This is a very similar question to Question 1a.

In [ ]:
is_ins_iid_unique = ...
is_ins_iid_unique

In [ ]:
grader.check("q3a")

<br/>

---

### 🔍 Question 3b

We want to extract `bid` from each row of the `ins` `DataFrame`. If we look carefully, the column `iid` of the `ins` `DataFrame` appears to be composed of two numbers, and the first number looks like a business ID.  

Create a new column called `bid` in the `ins` Dataframe containing just the business ID. You will want to use `.str` expressions. (The `str.split` expression could come in use; [read up on the documentation here](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.split.html)!) Also, be sure to convert the type of this column to `pl.Int64`. 

This question (3b) is a good candidate to try using an LLM to help you navigate the steps described above, but make sure you know exactly what's happening at each step!

**Hint**: Similar to an earlier problem where we used `cast(pl.String)` to convert a column to a string, here you should use `cast` to convert the `bid` column into type `pl.Int64`. `str.split` produces a list in each row; `.list.get(0)` grabs the first element. **No Python `for` loops or list comprehensions are allowed.**

In [ ]:
ins = ...
ins.head(5)

In [ ]:
grader.check("q3b")

<br/>

---

### 🔍 Question 3c (optional)

*This question is open-ended: we encourage you to try searching the documentation and/or using an LLM to help you.*

For this part, you should convert the `date` column of the `ins` table into a datetime type, and store the result in a column called `timestamp`. A good starting point is [this section of the user guide that describes the `.str.to_date()` method](https://docs.pola.rs/user-guide/transformations/time-series/parsing/#casting-strings-to-dates). **Be scrappy and resourceful!** You'll also need to figure out how to specify the format and date properly.

**No Python `for` loops or list comprehensions are allowed!**

In [ ]:
# Your code here to add a `timestamp` column to the `ins` table:
ins = ...

In [ ]:
grader.check("q3c")

<br/>

---

## 🏘️ 4: Search for the Perfect Restaurant

For this question, you work as a salesman. One of your clients is coming to San Francisco for a dinner meeting with you. You're in charge of picking the spot to eat. Your client wants to grab some food near Fisherman’s Wharf. They believe that the closer the restaurant is to the area the better the food will be. Your task is to use **coordinates** to choose the perfect restaurant for this dinner meeting to impress your important client.

<br/>

---

### 🏘️ Question 4a

Create a new `DataFrame` `bus_coords`, which includes only the rows of `bus_valid` where both the latitude and longitude values are not missing. 

**Hint:** Missing values for latitude and longitude values are the *same* as the most frequent missing value you found in **Question 2c**!

**Hint:** You do not need to consider the scenario where a phone number is missing!

In [ ]:
bus_coords = ...
bus_coords

In [ ]:
grader.check("q4a")

<br/>

---

### 🏘️ Question 4b

To be close to Fisherman’s Wharf, find the **three restaurants** in the `bus_coords` `DataFrame` that are furthest north, breaking ties by prioritizing restaurants that are furthest east. Return the `name`, `address`, and `postal5` (columns in that order) of the three restaurants as a `DataFrame` and assign it to `top3`. Here is an [article](https://www.britannica.com/science/latitude) that succinctly explains latitude and longitude.

**Hint:** Feel free to reference the [sort documentation](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html) and/or ask an LLM to see how you can sort by multiple values. It's up to you to figure out how or why this would be useful!

In [ ]:
# SCRATCH CELL
# Feel free to do your rough work here
# Do not add a cell between your solution and the grader cell

In [ ]:
top3 = ...
top3

In [ ]:
grader.check("q4b")

<!-- BEGIN QUESTION -->

##  📊 Part 2: Miscellaneous EDA Fun

## 🗣️ Question 5: Plot presentation

In their 2013 research paper titled [The Missing "One-Offs": The Hidden Supply of High-Achieving, Low-Income Students](https://www.brookings.edu/wp-content/uploads/2016/07/2013a_hoxby.pdf), Caroline Hoxby and Chris Avery investigate the behavior of high-achieving low-income applicants to undergraduate programs in the United States.

Here is Figure 10 of their research paper:

<img src="assets/hoxby-average-fig10.png" width="500" alt="Stacked histogram showing that high-achieving low-income students apply more often to selective colleges where their scores are far above the college median, while high-income students apply more often to colleges closer to their score level.">

**Your task**: Record a 60-90 second screencast describing the plot above.
-   If you need more than 60-90 seconds to record for accessibility reasons, please let the teaching staff know via Ed.

Guidelines:

-   Your audience is a UC Berkeley undergraduate who has not taken a statistics or data science course, and is unfamiliar with the research paper. You do not need to read the research paper in full to understand the plot, but you might find it helpful to reference.
-   In your screencast, you must explain the minimum necessary background required to understand the plot, along with the key takeaway(s) of the plot.
-   Your name and face should not be in the screencast. In other words, the plot should take up the entire window of the screencast, with your voice playing in the background. **Do not introduce yourself or use any identifying information.**
-   You should use your mouse pointer to indicate particular points of interest on the plot. Alternatively, you can verbally direct the viewer to points of interest (e.g., "In the top right corner, you can see that...").
-   You should use an engaging tone that sounds as though you are presenting to a live audience.
-   You are welcome to read off a script that you have written ahead of time, but try not to make it too obvious. In fact, it can be very helpful to write a script for a presentation ahead of time, even if you do not actually read the script when presenting.
- **LLM guidelines**:
  - Any words that you say or read aloud during your screencast must be your own, and must not be LLM-generated.
  - You are allowed to use an LLM to help you understand the paper, but we **strongly encourage** you to try reading and understanding most of the paper yourself rather than asking an LLM to summarize it.

**Remember the three key guiding questions you should address before digging into the details: (1) What's the meaning of the X axis (e.g., what does a -10 value mean)? (2) What's the meaning of the Y axis? (3) What does a specific point/line/feature on your plot mean in context?**

There are lots of free tools for recording screencasts. For example, QuickTime is a useful tool for recording screencasts on a Mac. Feel free to post on Ed if you cannot identify a way to record a screencast anonymously.

💾 Upload your screencast to your UC Berkeley Google Drive account, or another place that allows you to share a public link.

-   Before submitting, make sure your screencast is publicly accessible with the link. One easy way to do this: Open the link in an incognito window in Google Chrome.
-   If your video is private or unviewable when we try to access it, we will not be able to confirm that you submitted your video before the HW1 deadline. You may lose points and/or have to use slip day(s).

**Why complete this problem?** Communication is a critical, but often under-appreciated, component of the data science life cycle.
This exercise helps develop your data storytelling ability, which is essential for getting anyone to actually care about your statistical analyses!

Include a public link to your screencast here. **Make sure to test your public link in an incognito window before submission.** Broken links will lose credit.

**Optional**: If you would like detailed feedback from a member of course staff on your video response, please fill out [this form](https://forms.gle/QK5tubkLXS75xZJE9).

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br/><br/>

---
## 🤖 Question 6: How well do LLMs do as analysts with minimal instructions?

Ask Gemini **Thinking Mode** or another **thinking** LLM model to answer the question we posed in class: 

> Scenario: You are an analyst working for the UC Berkeley admissions office.
> Your manager asks you to identify five California public high schools that defy typical patterns of application, admission, and/or attendance.
> The admissions office wants to visit these schools to figure out what's going really well or really wrong."

If you'd like, feel free to push the LLM to go in different directions or provide it with additional data or tips. 

What do you think about the answer provided by the LLM? Here are some questions to think about:
- What are the strengths and weaknesses of the analysis provided by the LLM, relative to the analysis we covered in lecture? 
- Did the LLM do a good job of "showing its work"? 
- Did the LLM defend its choices against alternatives using quantitative and/or qualitative reasoning?

Feel free to write your answer in sentences or bullet points. Your response should be the equivalent of at least three sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/><br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Homework 1 Coding!

### Course Content Feedback

If you have any feedback about this assignment or about any of our other weekly, weekly assignments, lectures, or discussions, please fill out the [Course Content Feedback Form](https://docs.google.com/forms/d/e/1FAIpQLSfN9C-RJoe9hD8G2sbd1reJQh5H4WwKLFFrEt4DeQUBKmToJQ/viewform?usp=dialog). Your input is valuable in helping us improve the quality and relevance of our content to better meet your needs and expectations!

### Submission Instructions

Below, you will see one cell. Running this will automatically generate a zip file with your answers. Please submit this file to the Homework 1 Coding assignment on Gradescope. Gradescope will automatically submit the PDF from this file to the Homework 1 Coding Written assignment. **There is no need to manually submit Homework 1 Written answers; however, please check that the PDF was generated and submitted correctly**.

* **Homework 1 Coding**: Submit your Jupyter Notebook zip file for Homework 1 Coding,
which can be generated and downloaded from DataHub by using the `grader.export()`
cell provided below.
* **Homework 1 Coding Written**: Gradescope will automatically submit the PDF from the zip file submitted earlier. You do not need to submit anything to this assignment yourself, but please check that the submission went through properly.  

**You are responsible for ensuring your submission follows our requirements and that the automatic submission for Homework 1 Coding Written answers went through properly. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out to staff prior to the deadline.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)